In [ ]:
# this notebook calls functions in dimensionality_reduction module

In [ ]:
import os
import sys
from glob import glob

module_path = os.path.abspath(os.path.join('../module/'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from prostate_recurrence.data_exploration import dimensionality_reduction

In [ ]:
from glob import glob
import pandas as pd

## read data

### ... define paths to data that contain tile-level quantitative features

In [ ]:
slides = ['C001590_B02', 'C001578_B01']

In [ ]:
path_part_1 = '/Volumes/proj-sahai-tme-ml/working/processed_data/feature_engineering/clinical/prostate/chiip_cohort/slide/'

In [ ]:
path_part_2 = 'tile_size_1024/whole_slide/PSR/deconvolutions/psr/inverted_grayscale/tile_level_features/'

In [ ]:
data_file_pattern = 'features_out.csv'

### ... read 

In [ ]:
all_paths_to_data = []
for slide in slides:
    paths_to_data = glob(
        os.path.join(
            path_part_1,
            slide,
            path_part_2,
            "*",
            data_file_pattern
        )
    )
    all_paths_to_data.extend(
        paths_to_data
    )

In [ ]:
all_data_labeled = dimensionality_reduction.read_data(
    all_paths_to_data
)

### ... save raw data as .csv

In [ ]:
output_path_save_raw = os.path.join(
    "/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/20210831_outputs_test_analysis",
    "all_data_labeled.csv"
)

In [ ]:
all_data_labeled.to_csv(
    output_path_save_raw,
    index=False
)

## process data

### ... read raw data from .csv

In [ ]:
input_path_save_raw = os.path.join(
    "/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/20210831_outputs_test_analysis",
    "all_data_labeled.csv"
)
all_data_labeled = pd.read_csv(
    input_path_save_raw
)

### ... keep features from data

In [ ]:
features = all_data_labeled[
    [
        col
        for col in all_data_labeled.columns
        if col not in ["slide", "image_tile", "label"]
    ]
]
features.shape

### ... processing

In [ ]:
(
    processed_features,
    standardisation_attributes
) = dimensionality_reduction.process_data(features)

In [ ]:
processed_features.head(5)

In [ ]:
processed_data_labeled = pd.concat(
    [
        processed_features, 
        all_data_labeled[['slide', 'image_tile', 'label']]
    ],
    axis=1
)

### ... save processed data as .csv

In [ ]:
output_path_save_processed = os.path.join(
    "/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/20210831_outputs_test_analysis",
    "processed_data_labeled.csv"
)

In [ ]:
processed_data_labeled.to_csv(
    output_path_save_processed,
    index=False
)

### ... save transformation attributes

In [ ]:
output_path_save_transformation_attributes = os.path.join(
    "/Users/fux/Documents/Projects/DNN-prostate-ecm/2021m09__dimensionality_reduction/20210831_outputs_test_analysis",
    "processed_data_standardisation_attributes.csv"
)

In [ ]:
standardisation_attributes.to_csv(
    output_path_save_transformation_attributes,
    index=False
)

## PCA

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(
    n_components=min(
        processed_features.shape[0],
        processed_features.shape[1]
    )
)

In [ ]:
processed_features_pca = pca.fit_transform(processed_features)

In [ ]:
processed_features_pca.shape

In [ ]:
pca_components = pca.components_

In [ ]:
pca.components_[:, 0]

In [ ]:
evr = pca.explained_variance_ratio_

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
fig = plt.figure()
ax1 = fig.add_axes([0.2,0.2,0.7,0.7]); ax1.set_ylim(0,1)
ax2 = ax1.twinx(); ax2.set_ylim(0,1)
ax1.set_xticks(np.arange(1, evr.size+1, 5))
ax1.set_xlabel("Principal components")
ax1.set_ylabel("Fraction of variance explained", c = 'b')
ax2.set_ylabel("Cumulative fraction of variance explained", c = 'r')
ax1.bar(np.arange(1, evr.size+1), height=evr, color = 'b')
ax2.plot(np.arange(1, evr.size+1), np.cumsum(evr), c = 'r', marker = 'o', markerfacecolor='none')

In [ ]:
loadings = pca.components_[:2, :].T * np.sqrt(pca.explained_variance_[:2])

In [ ]:
unique_slide_names = processed_data_labeled['slide'].unique().tolist()
colorvals = np.linspace(0,1,len(unique_slide_names))
slide_names = processed_data_labeled['slide'].values

cs = [
    colorvals[unique_slide_names.index(slide_name)] 
    for slide_name in slide_names
]

fig = plt.figure(figsize = (4,3), dpi = 300)
ax1 = fig.add_axes([0.2,0.2,0.7,0.7]);
scatter = ax1.scatter(
    processed_features_pca[:, 0],
    processed_features_pca[:, 1],
    c = cs,
    s = 3
)
ax1.legend(
    handles=scatter.legend_elements()[0], 
    labels=unique_slide_names,
    prop={'size':6}
)
ax1.set_xlabel(f"PC1 ({round(evr[0]*100, 2)}% variance explained)")
ax1.set_ylabel(f"PC2 ({round(evr[1]*100, 2)}% variance explained)")


for i in range(loadings.shape[0]):
    ax1.plot(
        [0, loadings[i, 0]],
        [0, loadings[i, 1]]
    )